## Introduktion

#### inledning
Vi använde spotify recsys 1M playlists datasettet (https://www.aicrowd.com/challenges/spotify-million-playlist-dataset-challenge#challenge-dataset), gjorde inte challengen utan använde endast datan för att prova laga ett rekommendationssystem med ett intressant ämne/dataset eftersom vi båda använder spotify och har lite negativa åsikter om spotifys nuvarande rekommendationssystem. Vi lärde oss mycket nytt under projektets gång och därför var det rätt svårt att komma igång och ha en bra bild av vilka verktyg det lönar sig använda eftersom vi hittade dem "on the go".

#### tillvägagångssätt
Vi började med att tänka ut olika sätt som vi kan få rekommendationer med hjälp av detta dataset, det var lite svårt i början för att vi inte hade någon mera specifik information/data angående sångerna utan endast en massa playlists och deras innehåll. Det skulle ha varit enklare att komma på sätt om man skulle haft mera information per låt (t.ex. tempo, genre, instrument och tema etc.) vilket vi nog hittade data för men det var extremt många sånger som saknades så det blev inte av.

#### utmaningar
Extremt stort dataset jämfört med vad vi arbetat med tidigare. Detta kom med utmaningar, som t.ex. att hantera vad som laddas upp till RAM. Vi måste också radikalt minska mängden data som blev processerad p.g.a. problem med RAM vilket ledde till mycket crashande och hög tidsförbrukning. Ena av oss har även AMD grafikkort så inlärningsprocessen med cupy kunde inte göras på den datorn, det fungerar bara med NVIDIA grafikkort.

#### resultat
Till slut fick vi några fungerande sätt att rekommendera låtar, och vi vill t.o.m. påstå att de ger helt bra resultat speciellt med tanke på hur dåligt vi tycker att det nuvarande systemet är. Vårt är knappast bättre men man får helt vettiga rekommendationer.

Det finns definitivt mera avancerade sätt vi skulle kunna tillägga för att få bättre resultat. Vi skulle ha kunnat processera mera data.

#### hjälpmedel
- Dokumentation av pandas/sklearn/numpy
- https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.cosine_similarity.html#sklearn.metrics.pairwise.cosine_similarity
- ChatGPT

In [1]:
import json
import os
import pandas as pd
import numpy as np
from numpy import vstack
import time
from scipy.sparse import save_npz, find
from sklearn.metrics.pairwise import cosine_similarity
import cupy as cp
import cupyx.scipy.sparse as cp_sparse
import scipy.sparse as sp
from scipy.sparse import csc_matrix

## Importerar json-filerna

In [3]:
data_folder = '../data/'
data_frames = []
files_to_import = []

# Skapar en lista av alla datafiler den hittar
for filename in os.listdir(data_folder):
    if filename.startswith('mpd.slice.'):
        file_path = data_folder + filename
        files_to_import.append(file_path)
        if filename.endswith('200999.json'): #hur många filer som ska inkluderas
            print("done")
            break

# Sorterar listan
files_to_import = sorted(files_to_import, key=lambda x: (
    int(x.split(".")[4].split("-")[0]), x))

# Importerar listan
for file_path in files_to_import:
    print(file_path)
    with open(file_path, 'r') as file:
        data = json.load(file)

    playlist_data = data['playlists']
    data_frame = pd.DataFrame(playlist_data)
    # data_frame = data_frame.set_index('pid')

    data_frames.append(pd.DataFrame(data_frame))

data_frame = pd.concat(data_frames)

done
../data/mpd.slice.0-999.json
../data/mpd.slice.1000-1999.json
../data/mpd.slice.2000-2999.json
../data/mpd.slice.10000-10999.json
../data/mpd.slice.11000-11999.json
../data/mpd.slice.12000-12999.json
../data/mpd.slice.13000-13999.json
../data/mpd.slice.14000-14999.json
../data/mpd.slice.15000-15999.json
../data/mpd.slice.16000-16999.json
../data/mpd.slice.17000-17999.json
../data/mpd.slice.18000-18999.json
../data/mpd.slice.19000-19999.json
../data/mpd.slice.20000-20999.json
../data/mpd.slice.100000-100999.json
../data/mpd.slice.101000-101999.json
../data/mpd.slice.102000-102999.json
../data/mpd.slice.103000-103999.json
../data/mpd.slice.104000-104999.json
../data/mpd.slice.105000-105999.json
../data/mpd.slice.106000-106999.json
../data/mpd.slice.107000-107999.json
../data/mpd.slice.108000-108999.json
../data/mpd.slice.109000-109999.json
../data/mpd.slice.110000-110999.json
../data/mpd.slice.111000-111999.json
../data/mpd.slice.112000-112999.json
../data/mpd.slice.113000-113999.js

## Gör datat till pickle-format (sparar minne och håller kvar dataformat)

På grund av att datafilerna som vi jobbar med är flera stora json-filer kan vi konvertera dem till pickle-format (Den sparar datatyperna, en csv-fil skulle inte göra det) för att spara RAM och importera datafilerna i framtiden snabbare och lättare.

In [4]:
data_frame.to_pickle('../data/playlists.pkl')

## Importerar pickle-filen

In [2]:
data_frame = pd.read_pickle('../data/playlists.pkl')

## Skapar en lista av alla låtar och räknar hur många gånger en sång uppkommer


In [3]:
df = data_frame

track_lists = df['tracks'].tolist()

unique_tracks = []
processed_tracks_amount = 0
track_occurrences = {}

for track_list in track_lists:
    for track in track_list:
        track_uri = track.get('track_uri')
        artist_uri = track.get('artist_uri')
        track_name = track.get('track_name')
        artist_name = track.get('artist_name')

        if track_uri not in track_occurrences:
            unique_tracks.append(
                (track_uri, artist_uri, track_name, artist_name, 1))
            track_occurrences[track_uri] = 1
        else:
            track_occurrences[track_uri] += 1

        processed_tracks_amount = processed_tracks_amount + 1


# Lägger track_occurrences in på rätta plats
for i, (track_uri, artist_uri, track_name, artist_name, _) in enumerate(unique_tracks):
    if track_uri in track_occurrences:
        unique_tracks[i] = (track_uri, artist_uri, track_name,
                            artist_name, track_occurrences[track_uri])

# Konverterar till en dataframe
unique_tracks = pd.DataFrame(unique_tracks, columns=[
                             'track_uri', 'artist_uri', 'track_name', 'artist_name', 'occurrences'])
display(unique_tracks)

# Konverterar till csv
unique_tracks.to_csv('../data/track_data.csv')


,track_uri,artist_uri,track_name,artist_name,occurrences
0,spotify:track:0UaMYEvWZi0ZqiDOoHU3YI,spotify:artist:2wIVse2owClT7go1WT98tk,Lose Control (feat. Ciara & Fat Man Scoop),Missy Elliott,690
1,spotify:track:6I9VzXrHxO9rA9A5euc8Ak,spotify:artist:26dSoYclwsYLMAKD3tpOr4,Toxic,Britney Spears,1308
2,spotify:track:0WqIKmW4BTrj3eJFmnCKMv,spotify:artist:6vWDO969PvNqNYHIOW5v0m,Crazy In Love,Beyoncé,1673
3,spotify:track:1AWQoqb9bSvzTjaLralEkT,spotify:artist:31TPClRtHm23RisEBtV3X7,Rock Your Body,Justin Timberlake,911
4,spotify:track:1lzr43nnXAijIGYnCT8M8H,spotify:artist:5EvFsr3kj42KNv97ZEnqij,It Wasn't Me,Shaggy,2351
...,...,...,...,...,...
681800,spotify:track:3xLmarzSroQuXbTK44UXhD,spotify:artist:7LltPBlxmV5Ikukpns1IKG,Going Back,The Makepeace Brothers,1
681801,spotify:track:3ryw10oCE4NhNbanwrzurQ,spotify:artist:2dI9IuajQnLR5dLxHjTTqU,Conversations Between,The Polish Ambassador,1
681802,spotify:track:1kUoHfeBjoChgOPeBjFELn,spotify:artist:6MaRs3ljHDbWgRPOgs9kdy,Magic Chant,The Spirit of the Eagle,1
681803,spotify:track:37IrFeTPLJs1IKxoBDuN8b,spotify:artist:44zcDDVZOY0ck7KECNUPK1,Sansula (Max Cooper's Lost In Sound Mix),Dominik Eulberg,1


## Visar de 10 sånger som förekommer mest

In [4]:
unique_tracks = pd.read_csv('../data/track_data_100k.csv')

top_10_popular = unique_tracks.nlargest(10, 'occurrences')

display(top_10_popular[['track_name', 'artist_name', 'occurrences']])

,track_name,artist_name,occurrences
1335,HUMBLE.,Kendrick Lamar,4441
3713,One Dance,Drake,4240
1874,Closer,The Chainsmokers,4145
1396,Broccoli (feat. Lil Yachty),DRAM,4073
2715,Congratulations,Post Malone,3936
2701,Caroline,Aminé,3523
1333,iSpy (feat. Lil Yachty),KYLE,3476
1159,XO TOUR Llif3,Lil Uzi Vert,3456
3545,Location,Khalid,3439
2698,Bad and Boujee (feat. Lil Uzi Vert),Migos,3385


## Skapar en matris av hur många gånger en sång har varit i samma playlist med en annan sång

In [5]:
df = data_frame[0:10000]

# Kollar checkpoint-filen
try:
    with open('checkpoint.txt', 'r') as checkpoint:
        last_processed_pid = int(checkpoint.read())
        print('Checkpoint hittad: ' + str(last_processed_pid))
except FileNotFoundError:
    print('Checkpoint hittades inte')
    last_processed_pid = 0

track_indices = unique_tracks['track_uri'].tolist()

n = len(unique_tracks)

try:
    # Försöker ladda in matris-filen
    with cp.load('../data/co_occurrence_matrix_10k.npz') as loaded_npz:
        print('Datafil existerar')
        data = loaded_npz['data']
        indices = loaded_npz['indices']
        indptr = loaded_npz['indptr']
        shape = tuple(loaded_npz['shape'])
        co_occurrence_matrix = cp_sparse.csc_matrix(
            (data, indices, indptr), shape=shape)
except FileNotFoundError:
    print('Datafil existerar inte')

    # Skapa en ny matris om filen ej hittas
    co_occurrence_matrix = cp_sparse.coo_matrix((n, n), dtype=cp.float32)


playlists = df[['tracks', 'pid']]
processed_playlists = []

# display(playlists)

for index, row in playlists.iterrows():
    pid = row['pid']
    tracks = row['tracks']

    track_uris = [track['track_uri'] for track in tracks]

    processed_playlist = {'pid': pid, 'tracks': track_uris}
    processed_playlists.append(processed_playlist)


# display(processed_playlists)

track_uri_to_index = {track_uri: index for index,
                      track_uri in enumerate(track_indices)}
index_to_track_uri = {index: track_uri for index,
                      track_uri in enumerate(track_indices)}
# print(track_uri_to_index)
# print(index_to_track_uri)
start_pid = last_processed_pid

for playlist in processed_playlists:
    if playlist['pid'] <= start_pid:
        continue
    n_tracks = len(playlist['tracks'])
    data = cp.array([1] * (n_tracks * n_tracks), dtype=cp.float32)
    row_indices = cp.array([track_uri_to_index[track]
                            for track in playlist['tracks']
                            for _ in range(n_tracks)], dtype=cp.int32)
    col_indices = cp.array([track_uri_to_index[track]
                            for _ in range(n_tracks)
                            for track in playlist['tracks']], dtype=cp.int32)

    playlist_matrix = cp_sparse.coo_matrix(
        (data, (row_indices, col_indices)), shape=(n, n))

    # print('Playlist')
    # print(playlist_matrix)

    playlist_matrix.setdiag(0.0 * (playlist_matrix.data == 1))

    co_occurrence_matrix += playlist_matrix

    print(last_processed_pid)

    last_processed_pid += 1


Checkpoint hittad: 11199
Datafil existerar


## Laddar in co-occurrence-matrisen

In [6]:
with cp.load('../data/co_occurrence_matrix_10k.npz') as loaded_npz:  # 10k playlistor
    print('Datafil existerar')
    data = loaded_npz['data']
    indices = loaded_npz['indices']
    indptr = loaded_npz['indptr']
    shape = tuple(loaded_npz['shape'])
    co_occurrence_matrix = cp_sparse.csc_matrix(
        (data, indices, indptr), shape=shape)


Datafil existerar


## Hittar sångerna med hösta co-occurrence i allmänhet

In [7]:

# Använder find-funktionen för att hitta non-zero värden i sparse-matrisen
i, j, nonzero_values = find(co_occurrence_matrix.get())


co_occurrences = []

# Loopar igenom matrisen och kastar bort alla "diagonala" occurrences
for index in range(len(i)):
    row, col, value = i[index], j[index], nonzero_values[index]
    if row != col:
        co_occurrences.append((row, col, value))

# Sorterar co-occurrences y descending order
co_occurrences.sort(key=lambda x: x[2], reverse=True)

top_track_amount = 20

# Räkna ut vilka 2 sånger har högsta co-occurrence
top_tracks = []
for row, col, score in co_occurrences[:top_track_amount]:
    track1_uri = index_to_track_uri[row]
    track2_uri = index_to_track_uri[col]
    track1 = unique_tracks.loc[unique_tracks['track_uri'] ==
                               track1_uri].iloc[0]
    track2 = unique_tracks.loc[unique_tracks['track_uri'] ==
                               track2_uri].iloc[0]
    top_tracks.append((track1['track_name'], track1['artist_name'],
                      track2['track_name'], track2['artist_name'], score))


# Printa sångerna med hösta co-occurrence
for t1_name, t1_artist, t2_name, t2_artist, score in top_tracks:
    print(
        f"{t1_name} - {t1_artist}, {t2_name} - {t2_artist} Co-occurrence värde: {score}")


HUMBLE. - Kendrick Lamar, Congratulations - Post Malone Co-occurrence värde: 219.0
Congratulations - Post Malone, HUMBLE. - Kendrick Lamar Co-occurrence värde: 219.0
XO TOUR Llif3 - Lil Uzi Vert, HUMBLE. - Kendrick Lamar Co-occurrence värde: 211.0
HUMBLE. - Kendrick Lamar, XO TOUR Llif3 - Lil Uzi Vert Co-occurrence värde: 211.0
HUMBLE. - Kendrick Lamar, Mask Off - Future Co-occurrence värde: 207.0
Mask Off - Future, HUMBLE. - Kendrick Lamar Co-occurrence värde: 207.0
DNA. - Kendrick Lamar, HUMBLE. - Kendrick Lamar Co-occurrence värde: 192.0
HUMBLE. - Kendrick Lamar, DNA. - Kendrick Lamar Co-occurrence värde: 192.0
XO TOUR Llif3 - Lil Uzi Vert, Congratulations - Post Malone Co-occurrence värde: 182.0
Congratulations - Post Malone, XO TOUR Llif3 - Lil Uzi Vert Co-occurrence värde: 182.0
Bad and Boujee (feat. Lil Uzi Vert) - Migos, Bounce Back - Big Sean Co-occurrence värde: 177.0
Bounce Back - Big Sean, Bad and Boujee (feat. Lil Uzi Vert) - Migos Co-occurrence värde: 177.0
Broccoli (feat

## Räknar ut cosine similarity på matrisen

In [7]:
coo_matrix = csc_matrix(co_occurrence_matrix.get())

matrix = coo_matrix[:50000] # 50k blir 8.1GB stor fil
cosim_matrix = cosine_similarity(
    matrix, dense_output=False)

sp.save_npz(
    f'../data/cosim_matrix_50k.npz', cosim_matrix)


## Räknar ut de 10 mest liknande sånger för en sång
Använder cosine similarity matrisen för att räkna ut vilka sånger som förekommer oftast i samma playlists med en sång, och ger rekommendationer baserat på det.

In [5]:
cosim_matrix = sp.load_npz('../data/cosim_matrix_50k.npz')

track_indices = unique_tracks['track_uri'].tolist()

track_uri_to_index = {track_uri: index for index,
                      track_uri in enumerate(track_indices)}
index_to_track_uri = {index: track_uri for index,
                      track_uri in enumerate(track_indices)}


recommendation_amount = 10
track_uri = 'spotify:track:6041gM9Th1ViQcnzkIxnuv'

# Få sångens index
track_index = track_uri_to_index[track_uri]

# Får låtens similarity scores i relation till andra låtarna
similarity_scores = cosim_matrix[track_index].toarray().flatten()

# Sorterar dom baserat på similarity score
similar_track_indices = np.argsort(-similarity_scores)

# Konverterar indexerna till track_uri
similar_track_uris = [index_to_track_uri[i] for i in similar_track_indices]


seed = unique_tracks.loc[unique_tracks['track_uri'] ==
                         track_uri].iloc[0]

print(
    f"Rekommendera liknande sånger till: {seed['track_name']} av {seed['artist_name']}")
print("10 mest liknande sånger:")

artist_uris = []
for i, track_uri in enumerate(similar_track_uris[1:recommendation_amount+1], start=1):
    track = unique_tracks.loc[unique_tracks['track_uri'] == track_uri].iloc[0]
    similarity_score = similarity_scores[track_uri_to_index[track_uri]]
    print(
        f"{i}. {track['track_name']} by {track['artist_name']}, Similarity score: {similarity_score:.2f}")
    artist_uris.append(track['artist_uri'])

# Artist_uris från recommendationerna
unique_artist_uris = list(set(artist_uris))

Rekommendera liknande sånger till: Me Ama Me Odia av Ozuna
10 mest liknande sånger:
1. En La Intimidad by Ozuna, Similarity score: 0.92
2. Diles (feat. Arcangel, Nengo Flow, Dj Luian & Mambo Kings) by Farruko, Similarity score: 0.89
3. Tu No Vive Asi (feat. Mambo Kingz & DJ Luian) by Arcangel, Similarity score: 0.87
4. Soy Peor Remix (feat. J Balvin, Ozuna & Arcangel) by Bad Bunny, Similarity score: 0.87
5. Tu No Metes Cabra by Bad Bunny, Similarity score: 0.86
6. Sola (Remix) [feat. Daddy Yankee, Wisin, Farruko, Zion & Lennox] by Anuel Aa, Similarity score: 0.86
7. Ahora Dice by Chris Jeday, Similarity score: 0.85
8. Dile Que Tu Me Quieres by Ozuna, Similarity score: 0.85
9. Bebe (feat. Anuel AA) by Ozuna, Similarity score: 0.85
10. Cuatro Babys by Maluma, Similarity score: 0.84


## Rekommendationer baserade på en playlist

Vi väljer en playlist ur datan, sedan kollas vilka sånger och artister det finns i playlisten. Sedan baserat på det kan vi rekommendera mera sånger.

In [6]:
#lista på biisin i playlisten

playlistindex = 10

songs_in_playlist = []
for track in df.iloc[playlistindex]['tracks']:
    
    track_name = track.get('track_name')
    track_uri = track.get('track_uri')
    
    songs_in_playlist.append((track_uri, track_name))
    
#lista på artister i playlisten
artists = []
for track in df.iloc[playlistindex]['tracks']:

    artist_uri = track.get('artist_uri')
    artist_name = track.get('artist_name')
    
    artists.append((artist_uri, artist_name))

#lista på de 5 mest förekomna artisterna
artists = pd.Series(artists)

top5_artists = artists.value_counts().head(5)
top5_artists_list = top5_artists.index.tolist()

In [17]:
#Alla sånger av top 5 artisterna
newdf = pd.DataFrame()

artist_ID_values = [item[0] for item in top5_artists_list]
spotify_track_ID_values = [item[0] for item in songs_in_playlist]

for artist in artist_ID_values:
    selected_rows = unique_tracks[unique_tracks['artist_uri'] == artist]
    newdf = pd.concat([newdf, selected_rows], ignore_index=True)
    
#filtrera bort dom som redan finns
newdf = newdf[~newdf['track_uri'].isin(spotify_track_ID_values)]

## Vi kan ge rekommendationer baserat på vilka artister förekommer mest i playlisten

Vi kan kolla vilka artister det finns mest av i playlisten och sedan ge mera rekommendationer baserat på dem. Vi kan t.ex. kolla vilka sånger (som inte redan finns med i listan) är mest populära av de mest förekomna artisterna.

In [26]:
#10 mest populära sångerna av de 5 mest förekomna artisterna
display(newdf[newdf['artist_uri'].isin(artist_ID_values)].sort_values(by='occurrences', ascending=False).head(10))

,track_uri,artist_uri,track_name,artist_name,occurrences
556,spotify:track:6fujklziTHa8uoM5OQSfIo,spotify:artist:7iZtZyCzp3LItcw1wtPI3D,Black Beatles,Rae Sremmurd,3197
557,spotify:track:6mapJIPnQ23RTAevUoE0DL,spotify:artist:7iZtZyCzp3LItcw1wtPI3D,Swang,Rae Sremmurd,2528
334,spotify:track:0wdKiSBUT7aZkXUIdJWcwC,spotify:artist:1gPhS1zisyXr5dHTYZyiMe,2 Phones,Kevin Gates,1999
555,spotify:track:4jTiyLlOJVJj3mCr7yfPQD,spotify:artist:7iZtZyCzp3LItcw1wtPI3D,This Could Be Us,Rae Sremmurd,1847
297,spotify:track:10I3CmmwT0BkOVhduDy53o,spotify:artist:1gPhS1zisyXr5dHTYZyiMe,Really Really,Kevin Gates,1833
7,spotify:track:0z5ZPs57J2KERwM1tBM2GF,spotify:artist:55Aa2cqylxrFIXC767Z865,6 Foot 7 Foot,Lil Wayne,1787
5,spotify:track:4dASQiO1Eoo3RJvt74FtXB,spotify:artist:55Aa2cqylxrFIXC767Z865,"Sucker For Pain (with Wiz Khalifa, Imagine Dra...",Lil Wayne,1758
6,spotify:track:6ScJMrlpiLfZUGtWp4QIVt,spotify:artist:55Aa2cqylxrFIXC767Z865,A Milli,Lil Wayne,1576
470,spotify:track:3Q3myFA7q4Op95DOpHplaY,spotify:artist:2cFrymmkijnjDg9SS92EPM,do re mi,blackbear,1556
9,spotify:track:7tGlzXJv6GD5e5qlu5YmDg,spotify:artist:55Aa2cqylxrFIXC767Z865,Love Me,Lil Wayne,1204


### Vi kan även rekommendera t.ex. 3 stycken mest populära låtar per top5 artist. 

In [27]:
#5 artisternas 3 mest populära sånger

for artistID in artist_ID_values:
    display(newdf[newdf['artist_uri'] == artistID].sort_values(by='occurrences', ascending=False).head(3))


,track_uri,artist_uri,track_name,artist_name,occurrences
7,spotify:track:0z5ZPs57J2KERwM1tBM2GF,spotify:artist:55Aa2cqylxrFIXC767Z865,6 Foot 7 Foot,Lil Wayne,1787
5,spotify:track:4dASQiO1Eoo3RJvt74FtXB,spotify:artist:55Aa2cqylxrFIXC767Z865,"Sucker For Pain (with Wiz Khalifa, Imagine Dra...",Lil Wayne,1758
6,spotify:track:6ScJMrlpiLfZUGtWp4QIVt,spotify:artist:55Aa2cqylxrFIXC767Z865,A Milli,Lil Wayne,1576


,track_uri,artist_uri,track_name,artist_name,occurrences
334,spotify:track:0wdKiSBUT7aZkXUIdJWcwC,spotify:artist:1gPhS1zisyXr5dHTYZyiMe,2 Phones,Kevin Gates,1999
297,spotify:track:10I3CmmwT0BkOVhduDy53o,spotify:artist:1gPhS1zisyXr5dHTYZyiMe,Really Really,Kevin Gates,1833
336,spotify:track:6HMHgBHdLBQ0QYIaOp2gse,spotify:artist:1gPhS1zisyXr5dHTYZyiMe,I Don't Get Tired (#IDGT) [feat. August Alsina],Kevin Gates,935


,track_uri,artist_uri,track_name,artist_name,occurrences
470,spotify:track:3Q3myFA7q4Op95DOpHplaY,spotify:artist:2cFrymmkijnjDg9SS92EPM,do re mi,blackbear,1556
485,spotify:track:6y6jbcPG4Yn3Du4moXaenr,spotify:artist:2cFrymmkijnjDg9SS92EPM,Idfc,blackbear,1147
493,spotify:track:53mrVsi49rLHIaKBiSvElG,spotify:artist:2cFrymmkijnjDg9SS92EPM,do re mi (feat. Gucci Mane),blackbear,508


,track_uri,artist_uri,track_name,artist_name,occurrences
556,spotify:track:6fujklziTHa8uoM5OQSfIo,spotify:artist:7iZtZyCzp3LItcw1wtPI3D,Black Beatles,Rae Sremmurd,3197
557,spotify:track:6mapJIPnQ23RTAevUoE0DL,spotify:artist:7iZtZyCzp3LItcw1wtPI3D,Swang,Rae Sremmurd,2528
555,spotify:track:4jTiyLlOJVJj3mCr7yfPQD,spotify:artist:7iZtZyCzp3LItcw1wtPI3D,This Could Be Us,Rae Sremmurd,1847


,track_uri,artist_uri,track_name,artist_name,occurrences
600,spotify:track:70Ftfkea8wtVbVFj7UPjVM,spotify:artist:5G9kmDLg3OeUyj8KVBLzbu,These Days (Remix) [feat. Marcus Stroman],Mike Stud,133
604,spotify:track:2e3OgIbfZw5deCjLMGatSS,spotify:artist:5G9kmDLg3OeUyj8KVBLzbu,Frio,Mike Stud,119
594,spotify:track:4dYkT1n9UH87yhusky4oCT,spotify:artist:5G9kmDLg3OeUyj8KVBLzbu,Let Her Go (Remix),Mike Stud,98


### Vi kan ge rekommendationer baserat på en artists mest populära sånger

In [30]:
#En artists 10 mest populära sånger

display(newdf[newdf['artist_uri'] == 'spotify:artist:2cFrymmkijnjDg9SS92EPM'].sort_values(by='occurrences', ascending=False).head(10))

,track_uri,artist_uri,track_name,artist_name,occurrences
470,spotify:track:3Q3myFA7q4Op95DOpHplaY,spotify:artist:2cFrymmkijnjDg9SS92EPM,do re mi,blackbear,1556
485,spotify:track:6y6jbcPG4Yn3Du4moXaenr,spotify:artist:2cFrymmkijnjDg9SS92EPM,Idfc,blackbear,1147
493,spotify:track:53mrVsi49rLHIaKBiSvElG,spotify:artist:2cFrymmkijnjDg9SS92EPM,do re mi (feat. Gucci Mane),blackbear,508
487,spotify:track:13JyykwyYQ3T5QxxL34ukQ,spotify:artist:2cFrymmkijnjDg9SS92EPM,chateau,blackbear,434
471,spotify:track:1nMYtxDrONcoGnKRvxTwPv,spotify:artist:2cFrymmkijnjDg9SS92EPM,i miss the old u,blackbear,390
476,spotify:track:1XcXhu1X0xBFMDFyq0b7Ue,spotify:artist:2cFrymmkijnjDg9SS92EPM,4u,blackbear,369
486,spotify:track:0qgZvYWLIqDn0tXimF8gPa,spotify:artist:2cFrymmkijnjDg9SS92EPM,Dirty Laundry,blackbear,332
489,spotify:track:4hdog9vyyqG9pcppG2Izek,spotify:artist:2cFrymmkijnjDg9SS92EPM,90210 (feat. G-Eazy),blackbear,292
496,spotify:track:2Y4DwuII6ZzFZxmvD1s5o5,spotify:artist:2cFrymmkijnjDg9SS92EPM,Different Hos,blackbear,238
473,spotify:track:44UQpKpI54KOZLAFaDAr9X,spotify:artist:2cFrymmkijnjDg9SS92EPM,idfc (Tarro Remix),blackbear,217


## Co-occurence artister

Ge rekommendationer baserat på artisterna från co-occurence uträkningens resultat.

In [7]:
for artistID in unique_artist_uris:
    display(unique_tracks[unique_tracks['artist_uri'] == artistID].sort_values(by='occurrences', ascending=False).head(3))

,Unnamed: 0,track_uri,artist_uri,track_name,artist_name,occurrences
13998,13998,spotify:track:6NSMQFKgjpQb0KkjMDYIK0,spotify:artist:4SsVbpTthjScTS7U2hmr1X,Tu No Vive Asi (feat. Mambo Kingz & DJ Luian),Arcangel,90
14011,14011,spotify:track:2FIm6YsSGL5acOqSuJDh5s,spotify:artist:4SsVbpTthjScTS7U2hmr1X,Me Acostumbre (feat. Bad Bunny),Arcangel,74
3828,3828,spotify:track:1pLCpA1RN8avJxSD3ZCwhj,spotify:artist:4SsVbpTthjScTS7U2hmr1X,Pa' Que La Pases Bien,Arcangel,49


,Unnamed: 0,track_uri,artist_uri,track_name,artist_name,occurrences
5371,5371,spotify:track:2FrnTVSHjgnEylKGdHRUK1,spotify:artist:329e4yvIujISKGKz1BZZbO,"Diles (feat. Arcangel, Nengo Flow, Dj Luian & ...",Farruko,199
5370,5370,spotify:track:1lxswgIpzV6HhENRvkflES,spotify:artist:329e4yvIujISKGKz1BZZbO,Chillax,Farruko,197
5374,5374,spotify:track:2QBlGKAp4PquIGtA6iNAuJ,spotify:artist:329e4yvIujISKGKz1BZZbO,Lejos De Aquí,Farruko,181


,Unnamed: 0,track_uri,artist_uri,track_name,artist_name,occurrences
15042,15042,spotify:track:5q2JbCNi4FcnglgPfxcV65,spotify:artist:2R21vXR83lH98kGeO99Y66,"Sola (Remix) [feat. Daddy Yankee, Wisin, Farru...",Anuel Aa,147
15030,15030,spotify:track:4Zke60QJMlW60qnXmYLdC8,spotify:artist:2R21vXR83lH98kGeO99Y66,Sola,Anuel Aa,33
77034,77034,spotify:track:4J1e3hbf0e5YUZjkxyJWD3,spotify:artist:2R21vXR83lH98kGeO99Y66,La Última Vez,Anuel Aa,26


,Unnamed: 0,track_uri,artist_uri,track_name,artist_name,occurrences
13993,13993,spotify:track:5MT96Zz0ymUJNm8obKZQr0,spotify:artist:4q3ewBCX7sLwd24euuV69X,Soy Peor,Bad Bunny,145
5445,5445,spotify:track:4UG962ViiLqoUyx0RjCcwP,spotify:artist:4q3ewBCX7sLwd24euuV69X,"Soy Peor Remix (feat. J Balvin, Ozuna & Arcangel)",Bad Bunny,81
14019,14019,spotify:track:17nOaHsTn06jxuU9NpHuFU,spotify:artist:4q3ewBCX7sLwd24euuV69X,Pa Ti,Bad Bunny,67


,Unnamed: 0,track_uri,artist_uri,track_name,artist_name,occurrences
5416,5416,spotify:track:5u5MvmVtitax9R1Mxh3reC,spotify:artist:1i8SpTcr7yvPOmcqrbnVXY,Dile Que Tu Me Quieres,Ozuna,305
15026,15026,spotify:track:5zSOVwaY3sVjnmLhuEfsuf,spotify:artist:1i8SpTcr7yvPOmcqrbnVXY,Tú Foto,Ozuna,198
35259,35259,spotify:track:4vlbjBSMqycZk6t4HVRpnC,spotify:artist:1i8SpTcr7yvPOmcqrbnVXY,No Quiere Enamorarse (Remix) [feat. Daddy Yankee],Ozuna,106


,Unnamed: 0,track_uri,artist_uri,track_name,artist_name,occurrences
5440,5440,spotify:track:0qYTZCo5Bwh1nsUFGZP3zn,spotify:artist:1r4hJ1h58CWwUQe3MxPuau,Felices los 4,Maluma,451
5404,5404,spotify:track:6DUdDIRgLqCGq1DwkNWQTN,spotify:artist:1r4hJ1h58CWwUQe3MxPuau,Borro Cassette,Maluma,368
5442,5442,spotify:track:1iEwyiSLAunPR6uouANE0O,spotify:artist:1r4hJ1h58CWwUQe3MxPuau,El Perdedor,Maluma,198


,Unnamed: 0,track_uri,artist_uri,track_name,artist_name,occurrences
5419,5419,spotify:track:22eADXu8DfOAUEDw4vU8qy,spotify:artist:0qTZZWLzuD59Un5r1speHm,Ahora Dice,Chris Jeday,262
190779,190779,spotify:track:0Ba7G3iFQlBQ3xCIKpzDcB,spotify:artist:0qTZZWLzuD59Un5r1speHm,Dale Hasta Abajo,Chris Jeday,2


## Uppg. 3 Teoretiskt rekommendationssystem, en jämförelse.

Efter att ha kolla på alla resultat från tävlingen konstaterade vi att de var mycket advancerade speciellt med hur mycket inträning och resurser som behövdes för att fö systemen att fungera. Vi valde slumpmässigt dock 'A hybrid two-stage recommender system for automatic playlist continuation'.

Det var också rätt advancerat likasom alla andra tävlingsbidrag men vi tror att vi lyckades lista ut helheten och hur det fungerar. De ända liknelserna mellan våra lösningar var att de också använde sig av co-occurrence värden för att komma till rekommendationer, och att de använde .pkl filformat. Men kul hursomhelst att det fanns ens några liknelser till "proffsen".

#### Saker de gjorde bättre
De använde sig av .hdf filformat vilket gör att programmet kan läsa från skivan direkt och på så sätt använda mindre systemresurser (RAM). Vi har haft en hel del bluescreens på grund av att vår stackars 32GB RAM inte räckt till. De hade även en mycket kraftigare maskin med 56 threads och 256GB RAM, ändå tog deras inlärning ungefär 100 timmar, jämfört med vår 1 timme för co-occurrence matrisen med 10k playlists. cosine similarity uträkningen tog ungefär 10 minuter men filen blev 8GB stor och det tog upp all RAM.

De har definitivt haft en mycket bättre plan innan dom började med projektet. De visste troligen exakt hur dom skulle bygga upp projektet och vilka verktyg dom ska/kan använda medan vi hade en mycket mera "learn as we go" inställning.

Deras projekt var från vår synvinkel bra strukturerat och det märktes att de definitivt visste mera vad de höll på med.
Redan i deras publication var det en massa saker vi inte hade hört om förut, både matematiskt och teoretiskt, vilket gjorde det rätt svårläst.


#### Deras projektstruktur

- json_to_dataframe.ipynb - läser in datan
- validation_strategy.ipynb - preprocesserar datan för träning och validering
- lightfm.ipynb - tränar och evaluerar lightfm modellen
- lightfm_text.ipynb - träna ett rekommendationssystem för att rekommendera sånger baserat på nuvarande innehåll i -playlisten och playlistens namn
- candidate_selection.ipynb - sparar några olika kandidater, alltså sånger, om vi har tolkat det rätt.
- lightfm_features.ipynb - skapar lightfm features för inträningen, valideringen och test sets och lagrar dem i hdf format
- co_occurence_features.ipynb - skapar en co-occurence matris för sångerna i datan och skapar co-occurence features för ett givet set av playlists
- xgboost.ipynb - gör rekommendationerna